# 05 — Design Patterns

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/03_Advanced/05_design_patterns.ipynb)

## 📌 What is it?
Design patterns are proven, reusable solutions to common software design problems. Key patterns for AI engineers: Singleton, Factory, Observer, Strategy.

## 🧠 Why AI Engineers need this
LLM client libraries, model registries, experiment trackers, and plugin systems all implement these patterns.

In [ ]:
# ── SINGLETON — one instance only ──
class LLMClientPool:
    """Single shared pool of LLM API connections."""
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.connections = []
            cls._instance.call_count = 0
            print("Created LLMClientPool singleton")
        return cls._instance
    
    def call(self, prompt: str) -> str:
        self.call_count += 1
        return f"Response #{self.call_count} to: {prompt[:20]}"

pool1 = LLMClientPool()
pool2 = LLMClientPool()

print(pool1 is pool2)       # True — same object!
pool1.call("What is AI?")
print(pool2.call_count)     # 1 — shared state!

In [ ]:
# ── FACTORY — create objects without specifying exact class ──
from abc import ABC, abstractmethod

class BaseLLM(ABC):
    @abstractmethod
    def complete(self, prompt: str) -> str: ...

class GPT4(BaseLLM):
    def complete(self, prompt: str) -> str:
        return f"[GPT-4o] {prompt[:30]}..."

class Claude(BaseLLM):
    def complete(self, prompt: str) -> str:
        return f"[Claude] {prompt[:30]}..."

class Gemini(BaseLLM):
    def complete(self, prompt: str) -> str:
        return f"[Gemini] {prompt[:30]}..."

# Factory function
def create_llm(provider: str) -> BaseLLM:
    models = {"openai": GPT4, "anthropic": Claude, "google": Gemini}
    if provider not in models:
        raise ValueError(f"Unknown provider: {provider}")
    return models[provider]()

# Usage — caller doesn't need to know concrete class
llm = create_llm("anthropic")
print(llm.complete("What is machine learning?"))

In [ ]:
# ── STRATEGY — swap algorithms at runtime ──
from typing import Protocol

class ChunkStrategy(Protocol):
    def chunk(self, text: str) -> list[str]: ...

class FixedSizeChunker:
    def __init__(self, size: int = 500):
        self.size = size
    def chunk(self, text: str) -> list[str]:
        return [text[i:i+self.size] for i in range(0, len(text), self.size)]

class SentenceChunker:
    def chunk(self, text: str) -> list[str]:
        return [s.strip() for s in text.split('.') if s.strip()]

class RAGPipeline:
    def __init__(self, chunker: ChunkStrategy):
        self.chunker = chunker
    
    def process(self, text: str) -> list[str]:
        return self.chunker.chunk(text)

# Swap strategy without changing the pipeline
text = "First sentence. Second sentence. Third sentence."
rag = RAGPipeline(FixedSizeChunker(size=20))
print("Fixed:", rag.process(text)[:2])

rag.chunker = SentenceChunker()   # swap at runtime!
print("Sentence:", rag.process(text))

## 🔗 What's Next?
[06 — Memory Management →](06_memory_management.ipynb)